# 04 · Lambda Expressions, `map`, `filter`, `reduce` & Built-in Function Toolbox

**Module 05 – Functions** · Estimated time: 55 min

> Combines 📗 *Lambda-Expressions-Map-and-Filter*, 📗 *Built-in Functions* section (map, reduce, filter, zip, enumerate, all/any) and 📘 *Map, Filter, and Reduce*.

### 🎯 What you will learn
- Lambda (anonymous) functions: syntax, uses, limits
- `map()`, including with several iterables
- `filter()`, including `filter(None, ...)`
- `functools.reduce()` with and without an initializer
- `sorted()`, `min()`, `max()` with `key=`
- `all()`, `any()`, `sum()`, `zip()`, `enumerate()`
- `operator` module helpers
- When to prefer comprehensions

## 1. Lambda expressions

A **lambda** is a small, **anonymous**, one-expression function:

```python
lambda arguments: expression
```

Step-by-step from a normal function to a lambda (📗):

In [1]:
def square(num):
    result = num ** 2
    return result

def square(num):
    return num ** 2

def square(num): return num ** 2

square = lambda num: num ** 2     # the same function as a lambda
print(square(4))

16


In [2]:
even = lambda x: x % 2 == 0
first = lambda s: s[0]
rev = lambda s: s[::-1]
adder = lambda x, y: x + y
greet = lambda name="you": f"Hi {name}"
print(even(3), first("hello"), rev("hello"), adder(2, 3), greet())

False h olleh 5 Hi you


**Rules & tips**
- Body is a single **expression** (no statements like `if:` blocks, loops or `return`)
- Implicitly returns the expression's value
- ⚠️ PEP 8: don't *assign* lambdas to names (`f = lambda ...`); use `def` instead, which gives better tracebacks and a real name. Lambdas shine when passed **inline** to another function:

In [3]:
names = ["banana", "Apple", "cherry"]
print(sorted(names, key=lambda s: s.lower()))

['Apple', 'banana', 'cherry']


## 2. `map(function, iterable, ...)`

Applies a function to every item and returns a **lazy iterator**.

In [4]:
def square(num):
    return num ** 2

my_nums = [1, 2, 3, 4, 5]
print(map(square, my_nums))           # a map object (lazy)
print(list(map(square, my_nums)))     # force it into a list
for item in map(square, my_nums):
    print(item, end=" ")
print()

[1, 4, 9, 16, 25]
1 4 9 16 25 


In [5]:
def splicer(mystring):
    return "even" if len(mystring) % 2 == 0 else mystring[0]

names = ["Andy", "Eve", "Sally"]
print(list(map(splicer, names)))
print(list(map(lambda n: n ** 2, my_nums)))
print(list(map(str.upper, names)))       # methods work too
print(list(map(int, "1 2 3".split())))   # classic input-parsing idiom

['even', 'E', 'S']
[1, 4, 9, 16, 25]
['ANDY', 'EVE', 'SALLY']
[1, 2, 3]


### `map` with multiple iterables (📗 + 📘)

The function must accept that many arguments; stops at the shortest iterable.

In [6]:
a = [1, 2, 3, 4]
b = [5, 6, 7, 8]
c = [9, 10, 11, 12]
print(list(map(lambda x, y: x + y, a, b)))
print(list(map(lambda x, y, z: x + y + z, a, b, c)))
print(list(map(pow, [2, 3, 4], [3, 2, 1])))

[6, 8, 10, 12]
[15, 18, 21, 24]
[8, 9, 4]


In [7]:
# 📗 example: temperatures
def fahrenheit(c):
    return 9 / 5 * c + 32
temps = [0, 22.5, 40, 100]
print(list(map(fahrenheit, temps)))

[32.0, 72.5, 104.0, 212.0]


## 3. `filter(function, iterable)`

Keeps only the items for which the function returns a truthy value. Also lazy.

In [8]:
def check_even(num):
    return num % 2 == 0

nums = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(list(filter(check_even, nums)))
print(list(filter(lambda x: x % 2 == 0, nums)))
print([x for x in nums if x % 2 == 0])          # equivalent comprehension

[0, 2, 4, 6, 8, 10]
[0, 2, 4, 6, 8, 10]
[0, 2, 4, 6, 8, 10]


`filter(None, iterable)` keeps only **truthy** items (📘):

In [9]:
values = [0, 1, [], "hello", "", None, True, False]
print(list(filter(None, values)))

[1, 'hello', True]


## 4. `functools.reduce(function, iterable[, initializer])`

`reduce` is **not** a built-in in Python 3 (it was in Python 2); import it from `functools`. It repeatedly combines items **left to right** into a single value.

If `seq = [s1, s2, s3, ..., sn]`, then `reduce(f, seq)` computes:
```
step 1:  r = f(s1, s2)
step 2:  r = f(r,  s3)
...
final:   r = f(r,  sn)
```
For `[47, 11, 42, 13]` with `+`:
```
47 + 11 = 58 → 58 + 42 = 100 → 100 + 13 = 113
```
> ⚠️ **Old/broken in the original bootcamp:** the reduce notebook displayed a diagram from `http://www.python-course.eu/images/reduce_diagram.png`, an external image link that is unreliable and served over plain http. It's replaced by the text diagram above.

In [10]:
from functools import reduce
lst = [47, 11, 42, 13]
print(reduce(lambda x, y: x + y, lst))

max_find = lambda a, b: a if a > b else b
print(reduce(max_find, lst))              # (max() already does this!)
print(reduce(lambda x, y: x * y, [1, 2, 3, 4, 5]))   # product: 120

113
47
120


### With an initializer (📘)

In [11]:
print(reduce(lambda x, y: x + y, [], 0))       # 0: the initializer is the default for empty input
print(reduce(lambda acc, word: acc + len(word), ["hi", "hello"], 0))
try:
    reduce(lambda x, y: x + y, [])
except TypeError as e:
    print("TypeError:", e)

0
7
TypeError: reduce() of empty iterable with no initial value


### 📗 Classic exercise: digits → number

In [12]:
digits = [3, 4, 3, 2, 1]
print(reduce(lambda x, y: x * 10 + y, digits))    # 34321

34321


> 💡 Most reduces have a clearer built-in: `sum()`, `max()`, `min()`, `math.prod()`, `"".join()`, `any()`, `all()`. Reach for `reduce` only for custom aggregations.

## 5. `sorted()`, `min()`, `max()` with `key=`

In [13]:
people = [{"name": "Asha", "age": 31}, {"name": "Ravi", "age": 25}, {"name": "Meena", "age": 40}]
print(sorted(people, key=lambda p: p["age"]))
print(min(people, key=lambda p: p["age"])["name"])
print(max(["apple", "fig", "banana"], key=len))
print(max([], default="empty"))            # avoid ValueError on empty input

[{'name': 'Ravi', 'age': 25}, {'name': 'Asha', 'age': 31}, {'name': 'Meena', 'age': 40}]
Ravi
banana
empty


The `operator` module provides fast, readable alternatives to simple lambdas:

In [14]:
from operator import itemgetter, attrgetter, add, mul
print(sorted(people, key=itemgetter("age")))
pairs = [("b", 2), ("a", 2), ("c", 1)]
print(sorted(pairs, key=itemgetter(1, 0)))    # by number, then letter
print(reduce(mul, [1, 2, 3, 4]))

[{'name': 'Ravi', 'age': 25}, {'name': 'Asha', 'age': 31}, {'name': 'Meena', 'age': 40}]
[('c', 1), ('a', 2), ('b', 2)]
24


## 6. `all()` and `any()` (📗)

- `all(iterable)` → True if **every** item is truthy (True for empty)
- `any(iterable)` → True if **at least one** item is truthy (False for empty)

In [15]:
lst = [True, True, False, True]
print(all(lst), any(lst))
print(all([]), any([]))
scores = [72, 88, 95]
print(all(s >= 70 for s in scores), any(s > 90 for s in scores))
password = "Secr3t!x"
print(any(c.isdigit() for c in password) and any(c.isupper() for c in password))

False True
True False
True True
True


## 7. `zip()` and `enumerate()` (📗)

In [16]:
x = [1, 2, 3]
y = [4, 5, 6, 7]
print(list(zip(x, y)))                      # shortest wins
d1 = {"a": 1, "b": 2}
d2 = {"c": 4, "d": 5}
print(list(zip(d1, d2)))                    # zips the keys
print(list(zip(d2, d1.values())))
def switcharoo(d1, d2):
    return {k: v for k, v in zip(d1, d2.values())}
print(switcharoo(d1, d2))

[(1, 4), (2, 5), (3, 6)]
[('a', 'c'), ('b', 'd')]
[('c', 1), ('d', 2)]
{'a': 4, 'b': 5}


In [17]:
lst = ["a", "b", "c"]
for number, item in enumerate(lst):
    print(number, item)
months = ["March", "April", "May", "June"]
print(list(enumerate(months, start=3)))

0 a
1 b
2 c
[(3, 'March'), (4, 'April'), (5, 'May'), (6, 'June')]


How `enumerate` works internally (📗): it's a generator:
```python
def enumerate(sequence, start=0):
    n = start
    for elem in sequence:
        yield n, elem
        n += 1
```

## 8. More handy built-ins

In [18]:
print(sum([1, 2, 3], start=10))
print(abs(-5), round(2.567, 1), divmod(17, 5))
print(list(reversed("abc")), len({1, 2}))
print(isinstance(3, int), callable(len), callable(3))
print(list(range(3)), list(map(chr, range(97, 100))))

16
5 2.6 (3, 2)
['c', 'b', 'a'] 2
True True False
[0, 1, 2] ['a', 'b', 'c']


## 9. map/filter/reduce vs comprehensions (📘)

| Consider | Recommendation |
|---|---|
| Readability | Comprehensions are usually clearer for simple transforms/filters |
| Performance | Comprehensions are often slightly faster than `map`+`lambda` |
| Existing named function | `map(str, items)` / `filter(str.isdigit, items)` read nicely |
| Aggregation | `sum`, `max`, `min`, `math.prod` first; `reduce` for custom cases |
| Laziness | `map`/`filter`/generator expressions are lazy; list comprehensions are not |

## 📝 Summary
- `lambda args: expr` for short throwaway functions, mainly as `key=` arguments.
- `map` transforms, `filter` selects, `reduce` (from `functools`) aggregates.
- `sorted/min/max(..., key=...)`, `all`, `any`, `zip`, `enumerate` are everyday tools.
- Prefer comprehensions for readability.

## 🏋️ Practice (📗 Built-in Functions Assessment)
1. Use `map()` to create a function which finds the length of each word in a phrase.
2. Use `reduce()` to take a list of digits and return the number they represent: `[3, 4, 3, 2, 1]` → `34321`.
3. Use `filter()` to return words from a list that start with a target letter.
4. Use `zip()` and a comprehension to return a list of the same length where each value is two strings concatenated with a connector: `concatenate(['A','B'], ['a','b'], '-')` → `['A-a', 'B-b']`.
5. Use `enumerate()` to return a dict with list values as keys and indices as values: `d_list(['a','b','c'])` → `{'a': 0, 'b': 1, 'c': 2}`.
6. Use `enumerate()` to count how many items equal their index: `count_match_index([0,2,2,1,5,5,6,10])` → 4.

<details><summary>Show solutions</summary>

```python
from functools import reduce

def word_lengths(phrase):
    return list(map(len, phrase.split()))

def digits_to_num(digits):
    return reduce(lambda x, y: x * 10 + y, digits)

def filter_words(word_list, letter):
    return list(filter(lambda w: w[0] == letter, word_list))

def concatenate(L1, L2, connector):
    return [a + connector + b for a, b in zip(L1, L2)]

def d_list(L):
    return {value: i for i, value in enumerate(L)}

def count_match_index(L):
    return len([n for i, n in enumerate(L) if n == i])
```
</details>